# Setting Up Cribl Integration for Snowflake

The Cribl Pack for Snowflake will setup the REST collector to collect data from your Snowflake deployment. This will give you the option of using the JWT or OAUTH approach for authentication. For either path, the data will come in as a JSON object and the pipelines will be able to optimize the output for your environment.

Requirements
- Admin rights to your Snowflake environment
- Access to the Snowflake UI


In [ ]:
CREATE SECURITY INTEGRATION cribl_oauth
  TYPE = OAUTH
  ENABLED = TRUE
  OAUTH_CLIENT = CUSTOM
  OAUTH_CLIENT_TYPE = 'CONFIDENTIAL'
  OAUTH_REDIRECT_URI = 'https://localhost'
  OAUTH_ISSUE_REFRESH_TOKENS = FALSE
  OAUTH_ENFORCE_PKCE = FALSE;

In [ ]:
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.SESSIONS
WHERE CREATED_ON > DATEADD(day, -7, CURRENT_TIMESTAMP())
ORDER BY CREATED_ON ASC
LIMIT 10000;

In [ ]:
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.LOGIN_HISTORY
WHERE CREATED_ON > DATEADD(day, -7, CURRENT_TIMESTAMP())
ORDER BY CREATED_ON ASC
LIMIT 10000;

In [ ]:
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE CREATED_ON > DATEADD(day, -7, CURRENT_TIMESTAMP())
ORDER BY CREATED_ON ASC
LIMIT 10000;

In [ ]:
SELECT *
FROM SNOWFLAKE.ACCOUNT_USAGE.ACCESS_HISTORY
WHERE CREATED_ON > DATEADD(day, -7, CURRENT_TIMESTAMP())
ORDER BY CREATED_ON ASC
LIMIT 10000;

# JWT Token Approach (Optional)
The following session will help build out the JWT Token Approach. If you already have the OAUTH version working, you can stop here.

### First step is to generate the RSA Key Pair

#### Generate private key
`openssl genrsa 2048 | openssl pkcs8 -topk8 -inform PEM -out rsa_key.p8 -nocrypt`

#### Generate public key
`openssl rsa -in rsa_key.p8 -pubout -out rsa_key.pub`

In [ ]:
USE ROLE ACCOUNTADMIN;

CREATE USER IF NOT EXISTS cribl_collector
  TYPE = SERVICE
  DEFAULT_ROLE = CRIBL_COLLECTOR_ROLE
  DEFAULT_WAREHOUSE = COMPUTE_WH;

-- Assign the public key (paste contents without BEGIN/END header/footer)
ALTER USER cribl_collector SET RSA_PUBLIC_KEY='MIIBI...';

-- Grant access
CREATE ROLE IF NOT EXISTS CRIBL_COLLECTOR_ROLE;
GRANT IMPORTED PRIVILEGES ON DATABASE SNOWFLAKE TO ROLE CRIBL_COLLECTOR_ROLE;
GRANT USAGE ON WAREHOUSE COMPUTE_WH TO ROLE CRIBL_COLLECTOR_ROLE;
GRANT ROLE CRIBL_COLLECTOR_ROLE TO USER cribl_collector;

In [ ]:
#!/usr/bin/env python3
"""Generate a Snowflake JWT for key pair authentication."""
import hashlib
import base64
import time
import jwt  # pip install PyJWT cryptography
from cryptography.hazmat.primitives import serialization

# Configuration
ACCOUNT = "MYORG-MYACCOUNT"  # Uppercase account identifier
USER = "CRIBL_COLLECTOR"      # Uppercase username
PRIVATE_KEY_PATH = "rsa_key.p8"

# Load private key
with open(PRIVATE_KEY_PATH, "rb") as f:
    private_key = serialization.load_pem_private_key(f.read(), password=None)

# Get public key fingerprint
public_key_bytes = private_key.public_key().public_bytes(
    serialization.Encoding.DER,
    serialization.PublicFormat.SubjectPublicKeyInfo
)
sha256_hash = hashlib.sha256(public_key_bytes).digest()
fingerprint = "SHA256:" + base64.b64encode(sha256_hash).decode()

# Build JWT
now = int(time.time())
payload = {
    "iss": f"{ACCOUNT}.{USER}.{fingerprint}",
    "sub": f"{ACCOUNT}.{USER}",
    "iat": now,
    "exp": now + 3600,  # 60 minutes
}

token = jwt.encode(payload, private_key, algorithm="RS256")
print(token)

# Validate Cribl_Collector user
This step allows you to make sure the Cribl_Collector user was properly created and has the RSA key attached under the RSA_PUBLIC_KEY.

In [ ]:
DESC USER CRIBL_COLLECTOR;

# This query validates the Current Account and Organization.

In [ ]:
SELECT CURRENT_ACCOUNT(), CURRENT_ORGANIZATION_NAME();